# 3_4 — Object-held-out CV с доверительными интервалами (P0-a)

**Primary:** balanced **repeated** stratified grouped K-fold по объектам. Объекты распределяются с гарантией **CRR-объекта в каждом тест-фолде** и val из ≥2 объектов **обоих классов** (исправлены замечания о слабой стратификации/валидации). **Secondary:** LOO по объектам.

Неопределённость DPI-Flow на инференсе **пропагируется через conditional flow** (MC, `config.mc_samples_eval` сэмплов θ), а не берётся posterior mean.

> ⚠️ Сначала ПЕРЕматериализуйте `data/dataset` (ноутбуки 1_x, `prefix_strict_preonset=True`) и переобучите модели, иначе headline-числа невалидны (утечка префикса) и устаревший `config.json`.

Корректные CI считаются в 3_5 (**object-cluster bootstrap по pooled OOF**), не наивным √n_folds.

In [ ]:
import os, sys, json, time
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.exists(os.path.join(REPO, 'src')): REPO = os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'src')); os.chdir(REPO)
import numpy as np, pandas as pd
TABLES=os.path.join(REPO,'results','analysis_tables'); os.makedirs(TABLES, exist_ok=True)
PUB=os.path.join(REPO,'results','tables'); os.makedirs(PUB, exist_ok=True)
FIGS=os.path.join(REPO,'results','analysis_figs'); os.makedirs(FIGS, exist_ok=True)
print('REPO =', REPO)

In [ ]:
from liquefaction_ai import load_population_artifact
from liquefaction_ai.evaluation.cross_validation import build_folds, evaluate_fold, aggregate_cv, aggregate_object_conformal
import torch
device = torch.device('cpu')
QUICK = False        # True = demo-эпохи (дымовой тест)
N_SPLITS = 5
N_REPEATS = 3        # repeated CV (≥3) — требование рецензента; >1 усредняет повторы
pop, config = load_population_artifact('data/dataset')
meta = pop['meta']
# idempotency-штамп протокола (перезапуск перезаписывает, дублей нет)
meta_stamp = {'prefix_strict_preonset': getattr(config,'prefix_strict_preonset',None),
              'seed': config.seed, 'n_splits': N_SPLITS, 'n_repeats': N_REPEATS,
              'mc_samples_eval': getattr(config,'mc_samples_eval',1), 'ts': time.strftime('%Y-%m-%d %H:%M')}
json.dump(meta_stamp, open(os.path.join(TABLES,'cv_grouped_run_meta.json'),'w'), ensure_ascii=False, indent=2)
print('samples', len(meta), '| objects', meta['object'].nunique(), '|', meta_stamp)

## Primary: balanced repeated grouped CV (все сильные baselines)

In [ ]:
# NESTED=True → fold-local grid search для torch-моделей; CatBoost остаётся фиксированным
# заранее объявленным baseline-протоколом и не получает censored-aware N_liq/P3.
NESTED = True
folds = build_folds(meta, config, seed=42, loo=False, n_splits=N_SPLITS, n_repeats=N_REPEATS)
raw, samples = [], []
for gid, fold in enumerate(folds):
    r, s = evaluate_fold(pop, config, fold, gid, device, quick=QUICK, nested=NESTED)
    raw.append(r); samples.append(s)
    print(f"rep{fold['repeat']} fold{fold['fold']}: ", dict(zip(r.model, r.P3_Core.round(1))))
raw = pd.concat(raw, ignore_index=True); samples = pd.concat(samples, ignore_index=True)
raw.to_csv(os.path.join(TABLES,'cv_grouped_raw.csv'), index=False)        # overwrite (идемпотентно)
samples.to_csv(os.path.join(TABLES,'cv_grouped_samples.csv'), index=False)
summary = aggregate_cv(raw); summary.to_csv(os.path.join(TABLES,'cv_grouped_summary.csv'), index=False)
# Empirical site-held-out simultaneous coverage; это audit с object-bootstrap CI, не formal guarantee.
objconf = aggregate_object_conformal(samples, level=0.90)
objconf.to_csv(os.path.join(TABLES,'cv_object_conformal.csv'), index=False)
print('empirical site-held-out coverage@90:', {r['model']: (r['Coverage_emp'], (r['Coverage_lo'], r['Coverage_hi'])) for _,r in objconf.iterrows()})
print('models in CV:', sorted(raw.model.unique()))
summary[[c for c in summary.columns if c in ('model','n_folds') or c.endswith('_mean')][:12]]

> Наивный `*_ci95` в summary — только справочная дисперсия по фолдам. **Публикационные CI** — в 3_5 (object-cluster bootstrap).

## Secondary: leave-one-object-out (20 фолдов, тяжелее)

In [ ]:
RUN_LOO = True
if RUN_LOO:
    lf = build_folds(meta, config, seed=42, loo=True)
    rl, sl = [], []
    for gid, fold in enumerate(lf):
        r, s = evaluate_fold(pop, config, fold, gid, device, quick=QUICK, nested=NESTED); rl.append(r); sl.append(s)
    pd.concat(rl, ignore_index=True).to_csv(os.path.join(TABLES,'cv_loo_raw.csv'), index=False)
    pd.concat(sl, ignore_index=True).to_csv(os.path.join(TABLES,'cv_loo_samples.csv'), index=False)
    print('LOO done')
else: print('LOO пропущен')